In [37]:
import os
from dotenv import load_dotenv
import requests

load_dotenv()
API_TOKEN = os.getenv("API_TOKEN")

### Lichess bot account ID

In [24]:
resp = requests.get("https://lichess.org/api/account",
    headers={
      "Authorization": f"Bearer {API_TOKEN}"
    }
)

In [ ]:
if resp.status_code == 200:
    account_info = resp.json()
    print(account_info['id'])
else:
    raise


### Check upgrade to bot

In [ ]:
resp = requests.post(
    "https://lichess.org/api/bot/account/upgrade",
    headers={
      f"Authorization": "Bearer {API_TOKEN}"
    }
)

In [39]:
is_bot_account = resp.json()
is_bot_account


{'ok': True}

# find match

In [ ]:
time_control = 'rapid' # bullet blitz rapid classical
base_time = 900
increment = 10

cur_rating = 1500
rating_range = 100

batch_size = 3
is_rated = False

### Get challenges sent to account

In [ ]:
def choose_in_challenges(challenge_list_resp, time_control, cur_rating, rating_range, is_rated):
    challenge_list = challenge_list_resp.json()
    for challenge in challenge_list['in']:

        if challenge['speed'] == time_control and challenge['variant']['key'] == 'standard' and challenge['rated'] == is_rated:
            rating = challenge['challenger']['rating']

            if cur_rating - rating_range <= rating <= cur_rating + rating_range:
                return challenge['id']

    return ''

### find quality opponent

In [ ]:
import json

In [ ]:
def get_opponents(online_bots_resp, time_control: str, cur_rating: int, rating_range: int) -> list[str]:
    """returns id of possible opponents"""
    
    online_bots = online_bots_resp.text.split('\n')
    opponents = []

    for line in online_bots:
        bot = json.loads(line)
        rating = bot['perfs'][time_control]['rating']

        if cur_rating - rating_range <= rating <= cur_rating + rating_range:
            opponents.append(bot['id'])

    return opponents

### Challenge Opponents 

In [ ]:
import time
import random

In [ ]:
def challenge_opponent(opponent_id: str, is_rated: bool, base_time: int, increment: int):
    """
    base_time and increment are time control elements:
    Bullet: 1+0, 2+1
    Blitz: 3+0, 3+2, 5+0, 5+3
    Rapid: 8+0, 10+0, 15+10, 20+5,
    Classical: 25+0, 15+15, 30+20, 60+0
    """

    challenge_resp = requests.post(
            f"https://lichess.org/api/challenge/{opponent_id}",
            headers={
                "Content-Type": "application/x-www-form-urlencoded",
                "Authorization": f"Bearer {API_TOKEN}"
            },
            data={
                "clock.limit": str(base_time),
                "clock.increment": str(increment),
                "rated": "false" if not is_rated else "true",
                "color": "random",
                "variant": "standard",
                "fen": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
                "keepAliveStream": "true",
            }
        )

    challenge_resp = json.loads(challenge_resp.text.splitlines()[0])

    try:
        challenge_id = challenge_resp['id']
        print('challenge sent')
        return challenge_id
    except Exception:
        print(challenge_resp['error'])
        print('continuing to next opponent...')
        return ''

In [ ]:
def opponent_batches(opponent_ids: list[str], batch_size: int):
    random.shuffle(opponent_ids)    
    batch = []
    for opponent_id in opponent_ids:
        if len(batch) == batch_size - 1:
            batch.append(opponent_id)
            yield batch
            batch = []
        else:
            batch.append(opponent_id)
    if batch:
        yield batch

In [ ]:
def cancel_challenges(challenge_ids, game_id):
    for challenge_id in challenge_ids:
        if challenge_id and challenge_id != game_id:
            cancel_resp = requests.post(
                f"https://lichess.org/api/challenge/{challenge_id}/cancel",
                headers={
                    "Authorization": f"Bearer {API_TOKEN}"
                }
                ).json()
            try:
                cancel_successful = cancel_resp['ok']
                print('Challenge cancelled successfully')
            except Exception:
                print('Cancel challenge failed')

In [ ]:
# game id for games is the also the challenge id sent before the game is accepted

def get_game():
    challenge_list_resp = requests.get("https://lichess.org/api/challenge",
        headers={
            "Authorization": f"Bearer {API_TOKEN}"
        }
    )
    print('challenge_list request successful')

    game_id = choose_in_challenges(challenge_list_resp, time_control, cur_rating, rating_range, is_rated)
    in_challenge_successfull = False

    if game_id:
        in_challenge_successfull = requests.post(
            f"https://lichess.org/api/challenge/{game_id}/accept",
            headers={
            "Authorization": f"Bearer {API_TOKEN}"
            }
        )

        try:
            in_challenge_successfull = in_challenge_successfull['ok']
            print('successfully taken in challenge')
        except Exception:
            print('in challenge cancelled')

    if not game_id or not in_challenge_successfull:
        num_bot_resp = requests.get("https://lichess.org/api/bot/online?nb=512")
        print('online bots request successful')
        opponent_ids = get_opponents(num_bot_resp, time_control, cur_rating, rating_range)
        game_found = False

        for opponent_batch in opponent_batches(opponent_ids, batch_size=batch_size):
            challenge_ids = []

            for opponent_id in opponent_batch:
                challenge_id = challenge_opponent(opponent_id, is_rated=is_rated, base_time=base_time, increment=increment)
                challenge_ids.append(challenge_id)

            with requests.get("https://lichess.org/api/stream/event", headers={"Authorization": f"Bearer {API_TOKEN}"}, stream=True) as resp:
                keep_alive_count = 0

                for event in resp.iter_lines():
                    if event:
                        game = json.loads(event)

                        if game['type'] == 'gameStart':
                            game_id, color = game['game']['gameId'], game['game']['color']
                            game_found = True

                            cancel_challenges(challenge_ids, game_id)
                            break
                        
                    else:
                        print('waiting for response...')
                        keep_alive_count += 1
                        print('current count till next round: ' + str(5 - keep_alive_count))
                        if keep_alive_count >= 5:
                            
                            cancel_challenges(challenge_ids)
                            print('---Starting next round---')
                            break
        
            if game_found:
                break

    return game_id


In [ ]:
def pending_move(self):
    with requests.get(f"https://lichess.org/api/bot/game/stream/{self.game_id}",headers=self.headers, stream=True) as resp:

        for event in resp.iter_lines():
            if event:
                event = json.loads(event)

                try:
                    event_type = event['type']
                except KeyError:
                    raise RuntimeError("Wrong game ID or internet connection issues")

                if event_type == 'gameFull':
                    event = event['state']
                    event_type = 'gameState'

                if event_type == "gameState":
                    move_sequence: str = event['moves']
                    move= self.uci_interpreter.get_move_from_uci(move_sequence)
                    if isinstance(move, Move):
                        if event['status'] in END_STATUS:
                            self.game_present = False
                        return move
                    else:
                        raise RuntimeError("Unable to resolve uci -> " + move_sequence)
                        
                if event_type == "opponentGone":
                    if event['claimWinInSeconds'] <= 0:
                        self.game_present = False
                        requests.post(
                            f"https://lichess.org/api/bot/game/{self.game_id}/claim-draw",
                            headers=self.headers
                        )
                        return 

In [ ]:
get_game()